PASO 5.1| FEATURE ENGINEERING.

--------------------------------

NUESTRO MEJOR ALIADO PARA CAPTURAR UNA MEJOR CORRELACION LINEAL Y NO LINEAL ENTRE VARIABLES Y VARIABLE TARGET
ES ENTENDER LAS POSIBILIDADES QUE OFRECEN LOS DATOS DE LAS DIFRENTES DISTRIBUCIONES DE LAS VARIABLES
Y LAS CORRELACIONES LINEALES ENTRE VARIABLES GRACIAS AL MAPA DE CALOR.

SE TRATA DE VISUALIZAR CUANDO UNA DISTRIBUCION ES POSIBLE DERIVARLA EN DOS DADO SU SEGREGACION EN DOS O MAS GRANDES GRUPOS DE MUESTRAS.

TAMBIEN SE TRATA DE ENTENDER QUE CUANDO UNA VARIABLE TIENE MUCHA CORRELACION ENTRE OTRAS SE PUEDEN COMBINAR, INCLUSIO HACER MEDIAS ARITMETICAS Y SENSIBILIZAR LA VARIABLE PRODUCTO CAMBIANDOLA DE DISCRETA A CONTINUA.
BINARIZAR TAMBIEN ES UNA OPCION.

INCLUSO PASAR DE UNA VARIABLE CONTINUA A DISCONTINUA SI SU CORRELACION ES BAJA PARA CAPTURAR OTRAS CORRELACIONES NO LINEALES.

In [2]:
import numpy as np, random
import pandas as pd

In [3]:
df1 = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/UE128K.csv')


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


desired_size = 64000
proportion = desired_size / len(df1)
df2strat, _ = train_test_split(df1, train_size=proportion, stratify=df1['cnt_fc'], random_state=42)

numeric_cols = df2strat.select_dtypes(include=['number']).columns
df2strat[numeric_cols] = df2strat[numeric_cols].round(1)
for col in df2strat.columns:
    df2strat[col] = df2strat[col].fillna(df2strat[col].mode()[0])

print(f"Tamaño del DataFrame estratificado: {len(df2strat)}")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


desired_size = 32000
proportion = desired_size / len(df2strat)
df3strat, _ = train_test_split(df2strat, train_size=proportion, stratify=df2strat['cntgrp_fc'], random_state=42)

numeric_cols = df3strat.select_dtypes(include=['number']).columns
df3strat[numeric_cols] = df3strat[numeric_cols].round(1)
for col in df3strat.columns:
    df3strat[col] = df3strat[col].fillna(df3strat[col].mode()[0])

print(f"Tamaño del DataFrame estratificado: {len(df3strat)}")

In [ ]:
#Empezamos por binarizacion:
#Observamos distribuciones y variables con poca linealidad.
#Si es simetrica gaussiana se hace un cut, si es simetrica muy desplazada se binariza, ponderizando
#en funcion del diferencial de muestras de los dos primeros cuartiles respecto a los dos ultimos.
import pandas as pd

def factorizar_religiosidad(df, columna='rlgdgr'):

  
    def asignar_categoria(x):
        if x <= 3.9:
            return 0  # ateo
        elif 4 <= x <= 6:
            return 1  # escéptico
        else:
            return 2  # creyente

    df['rel3fc'] = df[columna].apply(asignar_categoria)
    return df

# Ejemplo de uso:
df1 = factorizar_religiosidad(df1)
print(df1['rel3fc'].value_counts())


In [ ]:
print(df1['happy'].value_counts())

In [ ]:
#18-'happy': Escala de 0 a 10, grado de cuan feliz eres, siendo 0 infeliz.
#Apenas tiene linealidad +/- vamos a derivarla en dos nuevas variables. A ver si capturamos correlaciones
# unhappybin y happybin
def pondbin(df, columna='happy'):

    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)

    def asignar_categoria(x):
        if x < q1:
            return 'bajo'
        elif x > q3:
            return 'alto'
        else:  # q1 <= x <= q3
            if x == 5:
                if abs(5 - q1) < abs(5 - q3):
                    return 'bajo'  # Ponderar hacia 'bajo'
                else:
                    return 'alto'  # Ponderar hacia 'alto'
            else:
                return 'medio'

    df['happyfc'] = df[columna].apply(asignar_categoria)
    return df
df1 = pondbin(df1)
print(df1['happyfc'].value_counts())


In [ ]:
dicthappy = {
    'bajo': '0',
    'medio': '1',
    'alto': '2',
}
df1['happyfc'] = df1['happyfc'].map(dicthappy)
print(df1['happyfc'].value_counts())

In [ ]:

def escalacontinuamedia(df):
    columnas_confianza = [
        'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trtsci_pnd'
    ]

    df['confianza_promedio'] = df[columnas_confianza].mean(axis=1)
    return df

df1 = escalacontinuamedia(df1)
print(df1['confianza_promedio'].describe())

In [ ]:

import pandas as pd
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '_factorizada'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'confianza_promedio', bins=5)
print(df1['confianza_promedio_factorizada'].value_counts())

In [ ]:
def escalacontinuamedia(df):
    columnas_satisfecho = [
        'stfgov', 'stfeco', 'stfdem',
    ]

    df['satisf_media'] = df[columnas_satisfecho].mean(axis=1)
    return df

df1 = escalacontinuamedia(df1)
print(df1['satisf_media'].describe())

In [ ]:
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '_factorizada'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'satisf_media', bins=5)
print(df1['satisf_media_factorizada'].value_counts())

In [ ]:
def pondbin(df, columna):

    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)

    def asignar_categoria(x):
        if x < q1:
            return 'bajo'
        elif x > q3:
            return 'alto'
        else:  # q1 <= x <= q3
            if x == 5:
                if abs(5 - q1) < abs(5 - q3):
                    return 'bajo'  # Ponderar hacia 'bajo'
                else:
                    return 'alto'  # Ponderar hacia 'alto'
            else:
                return 'medio'

    df['pintfc'] = df[columna].apply(asignar_categoria)
    return df
df1 = pondbin(df1, 'polintr')
print(df1['pintfc'].value_counts())

In [ ]:
dictint = {
    'bajo': '0',
    'medio': '1',
    'alto': '2',
}
df1['pintfc'] = df1['pintfc'].map(dicthappy)
print(df1['pintfc'].value_counts())

In [ ]:
def escalacontinuamedia(df):
    columnas_gente= [
        'pplfair', 'pplhlp', 'ppltrst',
    ]

    df['ppl'] = df[columnas_gente].mean(axis=1)
    return df

df1 = escalacontinuamedia(df1)
print(df1['ppl'].describe())

In [ ]:
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '_fc'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'ppl', bins=4)
print(df1['ppl_fc'].value_counts())

In [ ]:
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '_fc'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'lrscale', bins=4)
print(df1['lrscale_fc'].value_counts())

In [ ]:
def cutcontinua(df, columna, bins):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    if bins <= 0:
        raise ValueError("El número de bins debe ser mayor que 0.")

    labels = list(range(bins))  # Etiquetas numéricas (0, 1, 2, ...)
    df[columna + '2_fc'] = pd.cut(df[columna], bins=bins, labels=labels, right=False)
    return df

df1 = cutcontinua(df1, 'lrscale', bins=5)
print(df1['lrscale2_fc'].value_counts())

In [ ]:
df1.to_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/presplit.csv', index=False)

In [ ]:
import numpy as np
import pandas as pd
#Vamos a transformar escalas ordinarias de 0 a 10 a categoricas factorizadas de 3 clases, teniendo en cuenta el peso de la densidad en sus cuartilicos.

def ordinal_10_to_3(serie):


    # 1. Calcular cuartiles
    q1 = serie.quantile(0.25)
    q2 = serie.quantile(0.50)
    q3 = serie.quantile(0.75)

    # 2. Calcular densidad en el punto medio
    densidad_q2 = serie.between(q2 - 0.5, q2 + 0.5).mean()  # Ajusta el rango según sea necesario

    # 3. Calcular proporciones
    proporcion_primeros_cuartiles = serie[serie <= q2].count() / serie.count()
    proporcion_ultimos_cuartiles = serie[serie >= q2].count() / serie.count()

    def asignar_categoria(valor):
        if valor < q1:
            return 0  # Categoría inferior
        elif valor > q3:
            return 2  # Categoría superior
        else:  # Valor cerca del punto medio
            if np.random.rand() < densidad_q2:
                return 1  # Categoría central
            else:
                if np.random.rand() < proporcion_primeros_cuartiles / (proporcion_primeros_cuartiles + proporcion_ultimos_cuartiles):
                    return 0
                else:
                    return 2

    return serie.apply(asignar_categoria)

# Ejemplo de uso
serie_ordinal_10 = pd.Series(np.random.randint(1, 11, 1000))  # Serie de ejemplo
serie_ordinal_3 = ordinal_10_to_3(serie_ordinal_10)

print(serie_ordinal_3.value_counts())

Seguimos en otro documento para tantear en diferentes modelos.